# Long-read Quality Control


In [ ]:
# import necessary modules
import os
import re
import pandas as pd
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
import numpy as np

sns.set_style("ticks",{'axes.grid' : True})
sns.set_palette("colorblind")

plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["axes.titlepad"] = 20

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams["axes.titlesize"] = 30
plt.rcParams['axes.labelsize'] = 23.5
plt.rcParams['xtick.labelsize'] = 18
plt.rcParams['ytick.labelsize'] = 18
plt.rcParams['legend.fontsize'] = 18
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Liberation Sans']
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams["savefig.dpi"]=300


In [ ]:
input_nanopore_pre=list(snakemake.input.nanopore_pre)
input_nanopore_post=list(snakemake.input.nanopore_post)
input_pacbio_pre=list(snakemake.input.pacbio_pre)
input_pacbio_post=list(snakemake.input.pacbio_post)

SAMPLING=snakemake.params.sampling

output_summary_html=snakemake.output.summary_html
output_read_count_png=snakemake.output.read_count_png
output_read_count_svg=snakemake.output.read_count_svg
output_total_bases_png=snakemake.output.total_bases_png
output_total_bases_svg=snakemake.output.total_bases_svg
output_read_length_png=snakemake.output.read_length_png
output_read_length_svg=snakemake.output.read_length_svg
output_read_quality_png=snakemake.output.read_quality_png
output_read_quality_svg=snakemake.output.read_quality_svg


## Parse long-read QC statistics


In [ ]:
def clean_number(value):
	if pd.isna(value):
		return np.nan
	value=str(value).strip()
	value=re.sub("<[^<]+?>", "", value)
	value=value.replace(",", "")
	value=value.replace("bp", "")
	value=value.replace("reads", "")
	value=value.replace("bases", "")
	value=value.strip()
	if value in ["", "-"]:
		return np.nan
	try:
		return float(value)
	except ValueError:
		return np.nan

def parse_nanostats(path, platform, stage):
	values = {}
	with open(path) as handle:
		for line in handle:
			line=line.strip()
			if len(line)==0:
				continue
			line=re.sub("<[^<]+?>", "", line)
			if ":" in line:
				key, value = line.split(":", 1)
			elif "\t" in line:
				key, value = line.split("\t", 1)
			else:
				continue
			key = key.strip()
			value = value.strip()
			values[key] = value

	sample=os.path.basename(path)
	sample=sample.replace("_nanostats_preQC.html", "")
	sample=sample.replace("_nanostats_postQC.html", "")
	sample=sample.replace("_pacbio_nanostats_preQC.html", "")
	sample=sample.replace("_pacbio_nanostats_postQC_" + SAMPLING + ".html", "")

	out = {
		"sample": sample,
		"platform": platform,
		"stage": stage,
		"path": path
	}

	key_map = {
		"Number of reads": "number_reads",
		"Total bases": "total_bases",
		"Median read length": "median_read_length",
		"Mean read length": "mean_read_length",
		"Read length N50": "read_length_N50",
		"Mean read quality": "mean_read_quality",
		"Median read quality": "median_read_quality"
	}

	for raw_key, clean_key in key_map.items():
		out[clean_key] = clean_number(values.get(raw_key, np.nan))

	return out

rows=[]
for path in input_nanopore_pre:
	rows.append(parse_nanostats(path, "Nanopore", "Pre-QC"))
for path in input_nanopore_post:
	rows.append(parse_nanostats(path, "Nanopore", "Post-QC"))
for path in input_pacbio_pre:
	rows.append(parse_nanostats(path, "PacBio HiFi", "Pre-QC"))
for path in input_pacbio_post:
	rows.append(parse_nanostats(path, "PacBio HiFi", "Post-QC"))

qc_df=pd.DataFrame(rows)

if len(qc_df)==0:
	qc_df=pd.DataFrame(columns=["sample", "platform", "stage", "number_reads", "total_bases", "median_read_length", "mean_read_length", "read_length_N50", "mean_read_quality", "median_read_quality", "path"])

numeric_cols=["number_reads", "total_bases", "median_read_length", "mean_read_length", "read_length_N50", "mean_read_quality", "median_read_quality"]
for col in numeric_cols:
	if col in qc_df.columns:
		qc_df[col]=pd.to_numeric(qc_df[col], errors="coerce")

qc_df["total_Mbp"]=qc_df["total_bases"]/1000000
qc_df=qc_df.sort_values(["platform", "sample", "stage"])
qc_df


## Pre-QC vs Post-QC statistics


In [ ]:
pre_df=qc_df[qc_df["stage"]=="Pre-QC"].copy()
post_df=qc_df[qc_df["stage"]=="Post-QC"].copy()

compare_df=pre_df.merge(post_df, on=["sample", "platform"], suffixes=("_pre", "_post"), how="inner")

if len(compare_df)>0:
	compare_df["reads_kept_percent"]=compare_df["number_reads_post"]*100/compare_df["number_reads_pre"]
	compare_df["bases_kept_percent"]=compare_df["total_bases_post"]*100/compare_df["total_bases_pre"]
	compare_df["read_length_N50_change_percent"]=(compare_df["read_length_N50_post"]-compare_df["read_length_N50_pre"])*100/compare_df["read_length_N50_pre"]
	compare_df["mean_quality_change"]=compare_df["mean_read_quality_post"]-compare_df["mean_read_quality_pre"]
else:
	compare_df=pd.DataFrame(columns=["sample", "platform", "reads_kept_percent", "bases_kept_percent", "read_length_N50_change_percent", "mean_quality_change"])

compare_df=compare_df.sort_values(["platform", "sample"])
compare_df


In [ ]:
summary_cols=["sample", "platform", "stage", "number_reads", "total_Mbp", "mean_read_length", "median_read_length", "read_length_N50", "mean_read_quality", "median_read_quality"]
summary_cols=[col for col in summary_cols if col in qc_df.columns]
qc_summary=qc_df[summary_cols].copy()

if len(compare_df)>0:
	compare_summary=compare_df[["sample", "platform", "reads_kept_percent", "bases_kept_percent", "read_length_N50_change_percent", "mean_quality_change"]].copy()
	compare_summary["stage"]="Pre/Post comparison"
	compare_summary=compare_summary[["sample", "platform", "stage", "reads_kept_percent", "bases_kept_percent", "read_length_N50_change_percent", "mean_quality_change"]]
	html_out=qc_summary.style.background_gradient(cmap="RdYlGn").to_html()
	html_out=html_out + "<br><br><h2>Pre-QC vs Post-QC</h2>"
	html_out=html_out + compare_summary.style.background_gradient(cmap="RdYlGn").to_html()
else:
	html_out=qc_summary.style.background_gradient(cmap="RdYlGn").to_html()

with open(output_summary_html, "w") as fp:
	fp.write(html_out)


## Number of reads


In [ ]:
fig_width=max(8, 0.45*len(qc_df["sample"].unique()))
fig, ax = plt.subplots(figsize=(fig_width, 10))

plot_df=qc_df.dropna(subset=["number_reads"]).copy()
plot_df["number_reads_million"]=plot_df["number_reads"]/1000000

plot_df["stage"]=pd.Categorical(plot_df["stage"], categories=["Pre-QC", "Post-QC"], ordered=True)

sns.barplot(data=plot_df, x="sample", y="number_reads_million", hue="stage", hue_order=["Pre-QC", "Post-QC"], ax=ax)
ax.set_xlabel("Sample")
ax.set_ylabel("Number of reads (million)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="QC stage")

fig.savefig(output_read_count_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_read_count_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()



## Total bases


In [ ]:
fig_width=max(8, 0.45*len(qc_df["sample"].unique()))
fig, ax = plt.subplots(figsize=(fig_width, 10))

plot_df=qc_df.dropna(subset=["total_Mbp"]).copy()

plot_df["stage"]=pd.Categorical(plot_df["stage"], categories=["Pre-QC", "Post-QC"], ordered=True)

sns.barplot(data=plot_df, x="sample", y="total_Mbp", hue="stage", hue_order=["Pre-QC", "Post-QC"], ax=ax)
ax.set_xlabel("Sample")
ax.set_ylabel("Total bases (Mbp)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="QC stage")

fig.savefig(output_total_bases_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_total_bases_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()



## Read length N50


In [ ]:
fig_width=max(8, 0.45*len(qc_df["sample"].unique()))
fig, ax = plt.subplots(figsize=(fig_width, 10))

plot_df=qc_df.dropna(subset=["read_length_N50"]).copy()
plot_df["read_length_N50_kbp"]=plot_df["read_length_N50"]/1000

plot_df["stage"]=pd.Categorical(plot_df["stage"], categories=["Pre-QC", "Post-QC"], ordered=True)

sns.barplot(data=plot_df, x="sample", y="read_length_N50_kbp", hue="stage", hue_order=["Pre-QC", "Post-QC"], ax=ax)
ax.set_xlabel("Sample")
ax.set_ylabel("Read length N50 (kbp)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="QC stage")

fig.savefig(output_read_length_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_read_length_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()



## Mean read quality


In [ ]:
fig_width=max(8, 0.45*len(qc_df["sample"].unique()))
fig, ax = plt.subplots(figsize=(fig_width, 10))

plot_df=qc_df.dropna(subset=["mean_read_quality"]).copy()

plot_df["stage"]=pd.Categorical(plot_df["stage"], categories=["Pre-QC", "Post-QC"], ordered=True)

sns.barplot(data=plot_df, x="sample", y="mean_read_quality", hue="stage", hue_order=["Pre-QC", "Post-QC"], ax=ax)
ax.set_xlabel("Sample")
ax.set_ylabel("Mean read quality")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="QC stage")

fig.savefig(output_read_quality_png, format="png", bbox_inches="tight", transparent=True)
fig.savefig(output_read_quality_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()



## Pre-QC vs Post-QC retained reads and bases


In [ ]:
if len(compare_df)>0:
	plot_df=compare_df[["sample", "platform", "reads_kept_percent", "bases_kept_percent"]].copy()
	plot_df=plot_df.melt(id_vars=["sample", "platform"], var_name="metric", value_name="percent")
	plot_df["metric"]=plot_df["metric"].replace({"reads_kept_percent":"Reads kept", "bases_kept_percent":"Bases kept"})
	fig_width=max(8, 0.45*len(compare_df["sample"].unique()))
	fig, ax = plt.subplots(figsize=(fig_width, 10))
	sns.barplot(data=plot_df, x="sample", y="percent", hue="metric", ax=ax)
	ax.axhline(100, linestyle="--", linewidth=1, alpha=0.5)
	ax.set_xlabel("Sample")
	ax.set_ylabel("Post-QC / Pre-QC (%)")
	ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
	ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="")
	plt.show()
else:
	print("No paired Pre-QC/Post-QC samples available for comparison.")


## Pre-QC vs Post-QC N50 and quality change


In [ ]:
if len(compare_df)>0:
	plot_df=compare_df[["sample", "platform", "read_length_N50_change_percent", "mean_quality_change"]].copy()
	plot_df=plot_df.melt(id_vars=["sample", "platform"], var_name="metric", value_name="change")
	plot_df["metric"]=plot_df["metric"].replace({"read_length_N50_change_percent":"Read length N50 change (%)", "mean_quality_change":"Mean quality change"})
	fig_width=max(8, 0.45*len(compare_df["sample"].unique()))
	fig, ax = plt.subplots(figsize=(fig_width, 10))
	sns.barplot(data=plot_df, x="sample", y="change", hue="metric", ax=ax)
	ax.axhline(0, linestyle="--", linewidth=1, alpha=0.5)
	ax.set_xlabel("Sample")
	ax.set_ylabel("Change after QC")
	ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
	ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="")
	plt.show()
else:
	print("No paired Pre-QC/Post-QC samples available for comparison.")
